# Step 5 · From budget to bike share: the MLCC lookup table (Grenoble)

**Question:** if Grenoble invests *B* euros in total, what network does it get and what bike share does the model
predict — for every budget from 0 to 50 % of the cost of upgrading the whole network?

Running MLCC for each budget separately would be slow and could give inconsistent plans. Instead the budget grows
in **1000 equal steps**: at each step MLCC upgrades the network of the previous step with the new money plus any
money left unspent before. The result is a **lookup table** *budget → network metrics → predicted share*, used by the
long-term planning notebooks (step 7) and the comparison of step 6.

* Cost scenario (inspired by 2023 French public-works prices): **200 / 415 / 590 / 750 €/m** to bring an LTS 2 / 3 /
  4 / 5 street to LTS 1 (higher than step 4).
* The table is built in two stages: **(1)** MLCC → infrastructure metrics (slow, done once, resumable);
  **(2)** metrics → share with the step-3 model (instant; rerun it alone if the model coefficients change).
* Finally, smooth curves (polynomials, PCHIP, isotonic + PCHIP) are fitted to the stepwise table and compared by
  cross-validation, for use by continuous optimisers.

| | |
|---|---|
| **Input** | Grenoble 2026 network (step 1) |
| **Output** | `results/mlcc_lookup/` — `grenoble_mlcc_lookup_1000.csv`, `grenoble_lookup_with_mode_share.csv`, budget-response CSVs, surrogate curve |
| **Run time** | a few seconds if the table exists; ~5 min to rebuild it (`RECOMPUTE = True`) |

In [1]:
# Setup: find the repository root and make the `cyclelts` package importable.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cyclelts").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import hashlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

from cyclelts import config
from cyclelts.mode_share import GRENOBLE_MODEL, GRENOBLE_2026_SHARE
from cyclelts.optimization import UpgradeProblem, COSTS_LOOKUP, mlcc_lookup

N_POINTS = 1000                  # number of budget states (999 positive steps)
MAX_BUDGET_FRACTION = 0.50       # last state = 50 % of the full upgrade cost
CHECKPOINT_EVERY = 20
RECOMPUTE = False                # True = rebuild the table even if it exists
LOOKUP_DIR = config.LOOKUP_DIR
LOOKUP_DIR.mkdir(parents=True, exist_ok=True)
RAW_LOOKUP = LOOKUP_DIR / "grenoble_mlcc_lookup_1000.csv"
CHECKPOINT_PATH = LOOKUP_DIR / "mlcc_resume_checkpoint.pkl"
PARTIAL_CSV = LOOKUP_DIR / "lookup_partial.csv"

G0 = config.grenoble_graph(2026)
problem = UpgradeProblem(G0, COSTS_LOOKUP, GRENOBLE_MODEL)
B_MAX = problem.full_cost_all     # includes prohibited LTS 5 roads, which can never be bought
budget_grid = np.linspace(0.0, MAX_BUDGET_FRACTION * B_MAX, N_POINTS)
print(f"Baseline predicted share: {GRENOBLE_MODEL.share_of_graph(G0):.5f} %")
assert abs(GRENOBLE_MODEL.share_of_graph(G0) - GRENOBLE_2026_SHARE) < 0.005
print(f"B_max (full upgrade cost): €{B_MAX:,.0f} | last state: €{budget_grid[-1]:,.0f} | step: €{budget_grid[1]:,.0f}")

Baseline predicted share: 22.82022 %


B_max (full upgrade cost): €77,191,469 | last state: €38,595,734 | step: €38,634


## 1. Stage 1 — run MLCC over the 1000 cumulative budgets

Skipped if the table already exists (and `RECOMPUTE = False`). A checkpoint is written every 20 states, so an
interrupted run resumes where it stopped. The checkpoint is tied to the settings: change `LOOKUP_DIR` or delete
the checkpoint if you change the costs, the grid or the network.

In [3]:
signature = hashlib.sha256(repr((N_POINTS, MAX_BUDGET_FRACTION, B_MAX, COSTS_LOOKUP, G0.number_of_nodes(),
                                  G0.number_of_edges())).encode()).hexdigest()

def checkpoint(next_index, H, rows, spent):
    tmp = CHECKPOINT_PATH.with_suffix(".tmp")
    config.save_pickle(dict(config_sig=signature, next_index=next_index, graph=H, rows=rows, spent_eur=spent), tmp)
    tmp.replace(CHECKPOINT_PATH)
    pd.DataFrame(rows).to_csv(PARTIAL_CSV, index=False)

if RAW_LOOKUP.exists() and not RECOMPUTE:
    print("Lookup table already computed:", RAW_LOOKUP.relative_to(ROOT))
else:
    resume = dict(H_start=None, rows=None, start_idx=1, spent=0.0)
    if CHECKPOINT_PATH.exists() and not RECOMPUTE:
        saved = config.load_pickle(CHECKPOINT_PATH)
        if saved.get("config_sig") == signature:
            resume = dict(H_start=saved["graph"], rows=saved["rows"], start_idx=saved["next_index"], spent=saved["spent_eur"])
            print(f"Resuming at state {resume['start_idx']}/{N_POINTS-1}")
        else:
            raise ValueError("Checkpoint made with other settings: delete it or change LOOKUP_DIR.")
    rows, _ = mlcc_lookup(problem, budget_grid, checkpoint=checkpoint, checkpoint_every=CHECKPOINT_EVERY, **resume)
    pd.DataFrame(rows).sort_values("index").to_csv(RAW_LOOKUP, index=False)
    print("Saved", RAW_LOOKUP.relative_to(ROOT))

Lookup table already computed: results\mlcc_lookup\grenoble_mlcc_lookup_1000.csv


## 2. Stage 2 — predicted share from the saved metrics (no network computation)

`budget_allocated_eur` is the cumulative money made available; `actual_spent_eur` can be slightly lower because
MLCC stops at the first street it cannot afford (the rest is carried over to the next step).

In [4]:
metrics_df = pd.read_csv(RAW_LOOKUP).sort_values("budget_allocated_eur").reset_index(drop=True)
metrics_columns = ["index", "budget_allocated_eur", "budget_fraction", "actual_spent_eur", "unspent_eur",
                   "upgraded_edges", "cycling_length_km", "normalized_network_length", "weighted_lcc_normalized",
                   *[f"lcc_lts{g}_km" for g in range(1, 5)]]
metrics_df[metrics_columns].to_csv(LOOKUP_DIR / "grenoble_lookup_infrastructure_only.csv", index=False)

lookup_df = metrics_df[metrics_columns].copy()
lookup_df["indicator_I"] = (GRENOBLE_MODEL.w_length * lookup_df.normalized_network_length
                            + GRENOBLE_MODEL.w_lcc * lookup_df.weighted_lcc_normalized)
lookup_df["predicted_share_percent"] = GRENOBLE_MODEL.share_from_metrics(lookup_df.normalized_network_length,
                                                                         lookup_df.weighted_lcc_normalized)
lookup_df.to_csv(LOOKUP_DIR / "grenoble_lookup_with_mode_share.csv", index=False)
for column, name in [("predicted_share_percent", "budget_vs_mode_share.csv"),
                     ("normalized_network_length", "budget_vs_normalized_length.csv"),
                     ("weighted_lcc_normalized", "budget_vs_weighted_lcc.csv")]:
    lookup_df[["budget_allocated_eur", column]].to_csv(LOOKUP_DIR / name, index=False)

print(f"States: {len(lookup_df)} | baseline {lookup_df.predicted_share_percent.iloc[0]:.3f} % "
      f"-> {lookup_df.predicted_share_percent.iloc[-1]:.3f} % at €{lookup_df.budget_allocated_eur.iloc[-1]/1e6:.1f} M")
display(lookup_df[["budget_allocated_eur", "actual_spent_eur", "normalized_network_length",
                   "weighted_lcc_normalized", "predicted_share_percent"]].iloc[::111].round(4))

States: 1000 | baseline 22.820 % -> 39.791 % at €38.6 M


,budget_allocated_eur,actual_spent_eur,normalized_network_length,weighted_lcc_normalized,predicted_share_percent
0,0.000000e+00,0.000000e+00,1.0,0.4074,22.8202
111,4.288415e+06,4.287333e+06,1.0,0.5880,29.5563
222,8.576830e+06,8.558225e+06,1.0,0.6724,32.5387
333,1.286524e+07,1.283447e+07,1.0,0.7193,34.1080
444,1.715366e+07,1.714543e+07,1.0,0.7525,35.1702
555,2.144207e+07,2.143588e+07,1.0,0.7855,36.1855
666,2.573049e+07,2.571922e+07,1.0,0.8185,37.1550
777,3.001890e+07,3.001092e+07,1.0,0.8516,38.0807
888,3.430732e+07,3.430638e+07,1.0,0.8846,38.9602
999,3.859573e+07,3.859514e+07,1.0,0.9176,39.7910


## 3. Three investment-response curves

The normalized network length stays at 1: up to 50 % of `B_max`, MLCC only upgrades streets that are already
part of the cycling network (LTS 2–4); car-only LTS 5 roads, which would add length, come last in its order.
Connectivity and predicted share rise fast at first, then flatten: the first millions connect the existing LTS 1
islands, later money mostly improves streets that are already connected. The last table gives the order of
magnitude: about **+3.6 percentage points per M€ for the first million**, falling to ~1 pp/M€ over 10 M€.

In [5]:
for column, title, ylabel in [("predicted_share_percent", "Predicted bike share", "Predicted share (%)"),
                              ("normalized_network_length", "Normalized cycling-network length", "N (dimensionless)"),
                              ("weighted_lcc_normalized", "Weighted largest connected component", "C (dimensionless)")]:
    fig = go.Figure(go.Scatter(x=lookup_df.budget_allocated_eur/1e6, y=lookup_df[column], mode="lines", line=dict(width=2.5)))
    if column == "predicted_share_percent":
        fig.add_hline(y=50, line_dash="dot", annotation_text="50 % ceiling")
    fig.update_layout(template="plotly_white", height=420, title=f"Grenoble MLCC — {title}",
                      xaxis_title="Cumulative allocated budget (€ million)", yaxis_title=ylabel)
    fig.show()

# "Order of magnitude": average gain per million euros for the first B million
x, y = lookup_df.budget_allocated_eur.to_numpy(), lookup_df.predicted_share_percent.to_numpy()
C = lookup_df.weighted_lcc_normalized.to_numpy(); N = lookup_df.normalized_network_length.to_numpy()
mag = []
for B in [1, 5, 10, 20, x[-1]/1e6]:       # values linearly interpolated at B million euros
    gain = lambda v: (np.interp(B*1e6, x, v) - v[0]) / B
    mag.append({"Budget (€M)": B, "LCC gain per €M": gain(C), "Length gain per €M": gain(N),
                "Bike share gain (pp/€M)": gain(y)})
magnitudes = pd.DataFrame(mag)
magnitudes.to_csv(LOOKUP_DIR / "mlcc_response_magnitudes.csv", index=False)
display(magnitudes.round(4))

,Budget (€M),LCC gain per €M,Length gain per €M,Bike share gain (pp/€M)
0,1.0000,0.0971,0.0,3.6454
1,5.0000,0.0401,0.0,1.4915
2,10.0000,0.0285,0.0,1.0383
3,20.0000,0.0184,0.0,0.6514
4,38.5957,0.0132,0.0,0.4397


## 4. Smooth surrogates of the stepwise curve

Candidates: polynomials of degree 1–5, PCHIP (shape-preserving cubic) and isotonic regression + PCHIP. They are
compared with **five interleaved folds** (interpolation accuracy, not extrapolation). A surrogate is *valid* if it
never decreases and stays between the baseline and 50 %. The best valid one is exported.

In [6]:
from numpy.polynomial import Polynomial
from scipy.interpolate import PchipInterpolator
from sklearn.isotonic import IsotonicRegression

print("Downward steps in the MLCC curve:", int(np.sum(np.diff(y) < -1e-8)))
xmin, xmax = float(x.min()), float(x.max())
xn = (x - xmin) / max(xmax - xmin, 1.0)          # rescale budgets to [0, 1]
y_min = float(y[0])

def fit_predict(method, xa, ya, xt):
    order = np.argsort(xa); xa = np.asarray(xa)[order]; ya = np.asarray(ya)[order]
    if method.startswith("Polynomial "):
        return Polynomial.fit(xa, ya, deg=int(method.split()[-1]), domain=[0, 1])(xt)
    if method == "PCHIP":
        return PchipInterpolator(xa, ya, extrapolate=True)(xt)
    if method == "Isotonic + PCHIP":
        yiso = IsotonicRegression(increasing=True, out_of_bounds="clip").fit_transform(xa, ya)
        return PchipInterpolator(xa, yiso, extrapolate=True)(xt)
    raise ValueError(method)

methods = [f"Polynomial {d}" for d in range(1, 6)] + ["PCHIP", "Isotonic + PCHIP"]
xx = np.linspace(0, 1, 1201)
cv_rows, curves = [], {}
for method in methods:
    errors = []
    for fold in range(5):
        test = np.arange(len(xn)) % 5 == fold
        errors.extend((fit_predict(method, xn[~test], y[~test], xn[test]) - y[test]).tolist())
    e = np.array(errors)
    curve = np.asarray(fit_predict(method, xn, y, xx), float)
    curves[method] = curve
    cv_rows.append(dict(model=method, rmse_pp=float(np.sqrt(np.mean(e**2))), mae_pp=float(np.mean(np.abs(e))),
                        decreasing_intervals=int(np.sum(np.diff(curve) < -1e-7)),
                        below_baseline=bool(curve.min() < y_min - 1e-6), above_50=bool(curve.max() > 50 + 1e-6)))
cv_df = pd.DataFrame(cv_rows).sort_values("rmse_pp").reset_index(drop=True)
cv_df["shape_valid"] = (cv_df.decreasing_intervals == 0) & ~cv_df.below_baseline & ~cv_df.above_50
cv_df.to_csv(LOOKUP_DIR / "surrogate_model_validation.csv", index=False)
display(cv_df.round(5))
BEST_MODEL = cv_df[cv_df.shape_valid].iloc[0]["model"] if cv_df.shape_valid.any() else "Isotonic + PCHIP"
print("Best valid surrogate:", BEST_MODEL)

Downward steps in the MLCC curve: 0


,model,rmse_pp,mae_pp,decreasing_intervals,below_baseline,above_50,shape_valid
0,Isotonic + PCHIP,0.06116,0.00736,0,False,False,True
1,PCHIP,0.06116,0.00736,0,False,False,True
2,Polynomial 5,0.20592,0.08254,0,False,False,True
3,Polynomial 4,0.20884,0.09899,0,False,False,True
4,Polynomial 3,0.31180,0.22719,0,False,False,True
5,Polynomial 2,0.60473,0.47506,10,False,False,False
6,Polynomial 1,1.10052,0.84415,0,False,False,True


Best valid surrogate: Isotonic + PCHIP


In [7]:
budget_dense = xmin + xx * (xmax - xmin)
fig = go.Figure()
fig.add_scatter(x=x/1e6, y=y, mode="lines", name="MLCC lookup", line=dict(color="black", width=2))
for method in ["Polynomial 2", "Polynomial 3", "Polynomial 5", "PCHIP", "Isotonic + PCHIP"]:
    fig.add_scatter(x=budget_dense/1e6, y=curves[method], mode="lines", name=method,
                    line=dict(width=3 if method == BEST_MODEL else 1.5, dash="solid" if method == BEST_MODEL else "dash"))
fig.update_layout(template="plotly_white", height=550, title=f"MLCC lookup and smooth surrogates — selected: {BEST_MODEL}",
                  xaxis_title="Cumulative allocated budget (€ million)", yaxis_title="Predicted bike share (%)")
fig.show()

pd.DataFrame({"budget_allocated_eur": budget_dense, "fraction_Bmax": budget_dense / B_MAX,
              "surrogate_share_percent": np.clip(curves[BEST_MODEL], y_min, 50.0)}
             ).to_csv(LOOKUP_DIR / "surrogate_continuous_curve.csv", index=False)

def predict_share_from_cumulative_budget(budget_eur):
    """Surrogate share (%) for a cumulative budget inside the sampled range."""
    b = np.asarray(budget_eur, dtype=float)
    if np.any((b < xmin) | (b > xmax)):
        raise ValueError(f"Budget must be between €{xmin:,.0f} and €{xmax:,.0f}")
    return np.clip(fit_predict(BEST_MODEL, xn, y, (b - xmin) / (xmax - xmin)), y_min, 50.0)

print(f"Example: 25 % of B_max (€{0.25*B_MAX/1e6:.1f} M) -> {float(predict_share_from_cumulative_budget(0.25*B_MAX)):.3f} %")

Example: 25 % of B_max (€19.3 M) -> 35.683 %


### Cautions

* The 50 % ceiling comes from the calibrated model; clipping the surrogates is only a safeguard.
* The table describes **one** policy (sequential MLCC with carry-over of unspent money), not every possible plan.
* A smooth curve does not make street upgrades continuous: check any proposed budget against the nearby MLCC states.
* `B_max` includes roads that can never be upgraded, so 100 % of `B_max` is not reachable.
* All shares are model predictions (INSEE *Deux-roues total* calibration), not observed future shares.